Àlgebra Lineal Numèrica - Curs 2024/2025 - Óscar Rodríguez

---

# Normes vectorials i matricials

### Exemple Motivador: Matriu mal condicionada

Considerem el sistema lineal $A\boldsymbol{x} = \boldsymbol{b}$ amb:

$$
A =
\begin{pmatrix}
9 & 9 & 5 & 0\\
0 & 9 & 3 & 8\\
7 & 3 & 8 & 8\\
6 & 1 & 8 & 9
\end{pmatrix},\qquad
\boldsymbol{x} =
\begin{pmatrix}
x_1\\
x_2\\
x_3\\
x_4
\end{pmatrix}
,\qquad \boldsymbol{b} =
\begin{pmatrix}
23\\
20\\
26\\
24
\end{pmatrix},
$$

el qual té per solució $\boldsymbol{x} = (1,1,1,1)^\top$.

Observem que si modifiquem una mica el vector del terme independent, la tercera xifra significativa, $\bar{\boldsymbol{b}} = (22.9,20.1,25.9,24.1)^\top$, és a dir, considerem el sistema:

$$
\begin{pmatrix}
9 & 9 & 5 & 0\\
0 & 9 & 3 & 8\\
7 & 3 & 8 & 8\\
6 & 1 & 8 & 9
\end{pmatrix}
\begin{pmatrix}
\bar{x}_1\\
\bar{x}_2\\
\bar{x}_3\\
\bar{x}_4
\end{pmatrix}
=
\begin{pmatrix}
22.9\\
20.1\\
25.9\\
24.1
\end{pmatrix},
$$

ara te per solució:

$$
\bar{\boldsymbol{x}} =
\begin{pmatrix}
 118.8\\
 -14.7\\
-182.8\\
  87.6
\end{pmatrix}.
$$

&nbsp;

Fixem-nos que un canvi relativament petit ha produït un canvi extremadament gran en la solució!!

&nbsp;

Solucionem el problema amb numpy...

In [ ]:
import numpy as np
from scipy.linalg import cholesky, solve_triangular

# Matriu del sistema
A = np.array([[9,9,5,0],[0,9,3,8],[7,3,8,8],[6,1,8,9]],dtype = np.float64)

# Termes independents
b = np.array([23,20,26,24],dtype = np.float64)
b_e = np.array([22.9,20.1,25.9,24.1],dtype = np.float64)

# Solucionem el sistema amb el solver de numpy:
x = np.linalg.solve(A,b)
x_e = np.linalg.solve(A,b_e)

# Printem els resultats:
print("Solució sistema original amb numpy:\n",x)
print("\nSolució sistema modificat amb numpy:\n",x_e)


## Solucionem el sistema A^T A x = AT b via Cholesky

# 1) Calculem les matrius A^T A i el vector A^T b
AtA = A.T@A
Atb = A.T@b

# 2) Descomposició Cholesky de la matriu A^T A
L = cholesky(AtA, lower=True)
#L = np.linalg.cholesky(AtA) # També es pot fer amb cholesky de numpy

# 3) Resolem el sistem triangular inferior Ly = Atb per obtenir el valor de y
y = solve_triangular(L, Atb, lower=True)

# 4) Resolem el sistema triangular superior L^T x = y, per obtenir la solució x
x = solve_triangular(L.T, y)


print("\n\nSolució sistema AtA x = Atb:\n",x)

# 1) Calculem les matrius A^T A i el vector A^T b_e
AtA = A.T@A
Atb = A.T@b_e

# 2) Descomposició Cholesky de la matriu A^T A
L = cholesky(AtA, lower=True)
#L = np.linalg.cholesky(AtA) # També es pot fer amb cholesky de numpy

# 3) Resolem el sistem triangular inferior Ly = Atb per obtenir el valor de y
y = solve_triangular(L, Atb, lower=True)

# 4) Resolem el sistema triangular superior L^T x = y, per obtenir la solució x
x_e = solve_triangular(L.T, y)

print("\nSolució sistema AtA x_e = Atb_e:\n",x_e)

Solució sistema original amb numpy:
 [1. 1. 1. 1.]

Solució sistema modificat amb numpy:
 [ 118.8  -14.7 -182.8   87.6]


Solució sistema AtA x = Atb:
 [0.99999996 1.         1.00000006 0.99999997]

Solució sistema AtA x_e = Atb_e:
 [ 118.799997    -14.6999996  -182.79999532   87.5999978 ]


La solució del sistema modificat no s'assembla a l'original!!

In [ ]:
import sympy as sp
from sympy.solvers.solveset import linsolve

# Matriu del sistema
A = sp.Matrix([[9,9,5,0],[0,9,3,8],[7,3,8,8],[6,1,8,9]])

# Terme independnet modificat
b_e = sp.Matrix([sp.Rational(229,10),sp.Rational(201,10),sp.Rational(259,10),sp.Rational(241,10)])

# Definim el sistema:
system = A, b_e

# Solucionem el sistema:
linsolve(system)


{(594/5, -147/10, -914/5, 438/5)}

&nbsp;

Es tracta d'un **sistema mal condicionat**.

&nbsp;

I la pregunta natural és: **Com podem saber a priori si un sistema està mal condicionat?**

&nbsp;

## Normes vectorials:

##### **Definició:** Norma vectorial

Sigui $V$ un espai vectorial sobre un cos $\mathbb{K}$ amb $\mathbb{K}=\mathbb{R}$ o $\mathbb{K}=\mathbb{C}$. Diem que l'aplicació $\|\cdot\|: V \rightarrow \mathbb{R}$ és una **norma** en $V$ si verifica:

1.   No negativitat:
    $\displaystyle{||x||\geq 0\quad \forall x\in V}$. A més, $||x|| = 0 \Longleftrightarrow x = 0$.

2.   Homogeneïtat: $||kx|| = |k| \cdot||x||\quad \forall x\in V, \forall k\in \mathbb{K}$.

3.   Desigualtat triangular: $||x+y|| \leq ||x|| + ||y|| \quad \forall x,y\in V$.



##### *Exemples de normes:*

Són típiques les següents normes:

* *Norma 1*: $\displaystyle{\|x\|_1 = \sum_{i=1}^n |x_i|}$

* *Norma euclidiana*: $\displaystyle{\|x\|_2 = \sqrt{\sum_{i=1}^n |x_i|^2}}$

* *Norma inifinit, del suprem o del màxim*: $\displaystyle{\|x\|_\infty = \max_{i=1,\ldots,n} |x_i|}$

* *Norma* $p$ *de Hölder*: $\displaystyle{\|x\|_p = \left(\sum_{i=1}^n |x_i|^p\right)^{1/p}}$  (amb $p\geq 1$)

&nbsp;

**Observació:** La norma 1 ($p=1$) i la norma euclidiana ($p=2$) són casos particulars de normes Hölder.

In [11]:
## Espai per fer la implementació de les normes:

# No hi ha questionari pero les demnanarà al questionari 1

import numpy as np

# Norma p de Hölder

def holder(x, p):

    n = len(x)

    norma = 0

    for i in range(n):
        norma += abs(x[i]**p)
    
    norma**(1/p)

    return norma


def normaInfinit(x):

    return np.max(abs(x))



x = np.zeros(5)

for i in range(5):
    x[i] = i + 1;

print(x)

print(holder(x, 1))
print(holder(x, 2))

print(normaInfinit(x))






[1. 2. 3. 4. 5.]
15.0
55.0
5.0


### *Exercici 16*

*Considereu les normes de Hölder $\|\cdot\|_{p}$ ($p\ge 1$) i la norma del
màxim $\|\cdot\|_{\infty}$ en $\mathbb{C}^n$,
definides per a tot $x\in\mathbb{C}^n$ com:*

$$
	\| x\|_{p} = \left(\sum_{i=1}^{n} |x_{i}|^{p}\right)^{1/p},\qquad
	\|x\|_{\infty} = \max_{i = 1\div n}|x_{i}|.  
$$

*Demostreu que per a tot $x\in\mathbb{C}^n$,*

$$
	\lim_{p\to\infty}\|x\|_{p} = \|x\|_{\infty}.
$$

##### Resolució:

(A pissarra. Idea: Partim del límit + logaritme i exponencial)

&nbsp;

## Normes matricials:

##### **Definició:** Norma matricial

Una norma matricial és una norma en l'espai vectorial de les matrius $V = \mathcal{M}_{n}(\mathbb{K})$ amb $\mathbb{K} = \mathbb{R}$ o $\mathbb{K} = \mathbb{C}$ que a més és sub-multiplicativa, és a dir:

$$\|AB\| \leq \|A\|\,\|B\|, \quad \forall A,B\in E$$

&nbsp;

### Norma matricial induïda i radi espectral

##### **Definició:** Norma matricial consistent

Direm que una norma matricial és consistent amb una norma vectorial (que de notarem igual) si:

$$
\| Ax\| \leq \|A\|\,\|x\|,\quad \forall A\in \mathcal{M}_n(\mathbb{K}), \forall x\in \mathbb{K}^n.
$$

&nbsp;

##### **Definició:** Norma matricial induïda

Siguin $\|\cdot\|_\alpha$ una norma vectorial a $\mathbb{K}^n$ i $\|\cdot\|_\beta$ una norma vectorial a $\mathbb{K}^m$, amb $\mathbb{K} = \mathbb{R}$ o $\mathbb{K} = \mathbb{C}$.
Per qualsevol matriu $A\in \mathcal{M}_{m\times n}(\mathbb{K}$) podem definir la seva norma matricial induïda com

$$
    \| A\|_{\alpha,\beta} = \sup_{x\,\in\,\mathbb{K}^n\setminus\{0\}} \frac{\|Ax\|_\beta}{\|x\|_\alpha} = \sup_{\|x\|_{\alpha} = 1} \|Ax\|_\beta .
$$

En el cas d'utilitzar la mateixa norma vectorial per $\mathbb{K}^n$ i $\mathbb{K}^m$, podem definir

$$
\|A\| := \sup_{x\neq 0} \frac{\|Ax\|}{\|x\|} = \sup_{\|x\| = 1} \| Ax\|.
$$

Aquest cas també es coneix com norma matricial subordinada.


**Observació:** És senzill veure que les normes matricials induïdes són consistent.

##### *Exemples*

* $\displaystyle{||A||_1 = \max_{1\leq j\leq n} \sum_{i=1}^n |a_{ij}| }$ (màxim de la suma en valor absolut dels elements de les columnes)

* $\displaystyle{||A||_\infty = \max_{1\leq i\leq n} \sum_{j=1}^n |a_{ij}| }$ (màxim de la suma en valor absolut dels elements de les files)

* $\displaystyle{||A||_2 = \sqrt{\rho(A^\top A)}}$ (arrel del màxim valor propi de $A^\top A$, pel cas de matrius a coeficients reals, pel cas de matrius complexes hem de calcular el VAP màxim de $A^*A$, sent $A^*$ la conjugada transposada).

##### **Definició:** Radi espectral $\rho(A)$ d'una matriu

Si $A$ és una matriu $n\times n$ amb valors propis $\lambda_1,\ldots,\lambda_s$, aleshores el seu radi espectral $\rho(A)$ es defineix per

$$
    \rho(A) := \max_{i} |\lambda_i|.
$$

&nbsp;

##### **Lema:** Fita radi espectral

Sigui $A\in\mathcal{M}_n(\mathbb{C})$, aleshores

$$
\rho(A) \leq \|A^k\|^{1/k}, \quad \forall k\in\mathbb{N}
$$

i per qualsevol norma matricial.

##### Demostració:

Per veure-ho, utilitzarem que donat un valor propi $\lambda$ de la matriu $A$ i el seu vector propi associat $v$ se satisfà:

$$A^kv = \lambda^k v.$$

Així:

$$
|\lambda|^k\,\|v\| = |\lambda^k|\, \|v\| = \|\lambda^k v\| = \| A^k v\| \leq \|A^k\|\,\|v\|,
$$

i, per tant:

$$
|\lambda|^k \leq \|A^k\|.
$$
o, equivalentment:
$$
|\lambda| \leq \|A^k\|^{1/k}.
$$


Com això es satisfà per a tot valor propi $\lambda$ de $A$, i $\displaystyle{\rho(A) = \max_i |\lambda_i|}$ tenim

$$
\rho(A) \leq \|A^k\|^{1/k}
$$

$\square$.

##### **Corol·lari:** Exercici 17a.

&nbsp;

##### **Observació:** No totes les normes matricials són induïdes

###### *Exemple: (Exercici 15)*

* $\displaystyle{||A||_F = \sqrt{\sum_{i=1}^m\sum_{j=1}^n|a_{ij}|^2} = \sqrt{\text{tr}(A^\top A)}}$ (Norma de Frobenius: arrel de la traça de $A^\top A$, pel cas de matrius a coeficients reals, pel cas de matrius complexes hem de calcular la traça de $A^*A$, sent $A^*$ la conjugada transposada).

### *Exercici 14*

*Demostreu que les normes matricials subordinades  a les normes vectorials*

$$
\| x \|_1 =\sum_i |x_i| \ , \quad  \| x \|_{\infty}=\max_i
|x_i|\ ,
$$

*són, respectivament:*

$$
\| A \|_1 =\max_j \sum_i |a_{ij}| \ , \quad  \| A
\|_{\infty}=\max_i\sum_j |a_{ij}| \, .
$$

##### Resolució: (Veure *Útiles básicos de Cálculo Numérico*. Aubanell, Benseny, Delshams)

Partim de la definició de $||Ax||_\infty$:

$$
\begin{aligned}
\|Ax\|_\infty &= \max_{i} \left|\sum_{j} a_{ij}x_j\right| & \quad (\text{Definició})\\
& \leq \max_{i} \left\{\sum_j |a_{ij} x_j|\right\} &\quad (\text{Suma valors absoluts})\\
& = \max_{i} \left\{\sum_j |a_{ij}|\cdot| x_j|\right\} &\quad (\text{Multiplicació valors absoluts})\\
& \leq \max_{i} \left\{\sum_j |a_{ij}|\cdot\|x\|_\infty\right\} &\quad (|x_j| \leq \max_{k} |x_k| = \|x\|_\infty)\\
& = \|x\|_\infty \cdot \max_{i} \left\{\sum_j |a_{ij}|\right\} & \quad (\text{factor comú})
\end{aligned}
$$

i, per tant:

$$
\| A\|_\infty \leq \max_{i} \left\{\sum_j |a_{ij}|\right\}
$$

&nbsp;

Triem ara el $k$ tal que

$$
\sum_j |a_{kj}| = \max_{i} \left\{\sum_j |a_{ij}|\right\},
$$

i escollim $\bar{x}$ definit per $\bar{x}_j = \text{sgn}(a_{kj})$. D'aquesta forma, tenim que $||\bar{x}||_\infty = 1$ i

$$
\begin{aligned}
\|A\bar{x}\|_\infty &= \max_{i} \left|\sum_{j} a_{ij}\bar{x}_j\right| & \quad (\text{Definició})\\
&= \max_{i} \left|\sum_{j} a_{ij} \cdot\text{sgn}(a_{kj})\right| & \quad (\bar{x}_j = \text{sgn}(a_{kj}))\\
&= \sum_j |a_{kj}| &\quad (\text{màxim})\\
&= \max_{i} \left\{\sum_j |a_{ij}|\right\}
\end{aligned}
$$

Així:

$$
||A||_\infty = \max_{i} \left\{\sum_j |a_{ij}|\right\}.
$$

&nbsp;

&nbsp;

## Nombre condició i propagació de l'error

#### **Definició:** Nombre de condició

Sigui una matriu $A \in \mathcal{M}_{n} (\mathbb{K})$ amb $\mathbb{K} = \mathbb{R}$ o $\mathbb{K} = \mathbb{C}$, una matriu invertible. Diem que el seu número de condició $\mu(A)$ és

$$
 \mu(A) = \| A\|\cdot\|A^{-1}\|.
$$

&nbsp;

**Observació 1:** Fixem-nos que el seu valor dependrà de la norma que utilitzem.

&nbsp;

**Observació 2:** Un pot veure fàcilment que $\mu(A) \geq 1.$

&nbsp;

**Observació 3:** Típicament no és calcula la matriu inversa, si no que es pot calcular el nombre de condició a la vegada que es fa la descomposició.

&nbsp;

#### Exemple:

Pel cas de la norma matricial induïda per la norma euclidiana el nombre de condició ve donat per:

$$
\mu_2(A) = \frac{|\lambda_{\text{max}}|}{|\lambda_{\text{min}}|} ,
$$

sent $\lambda_{\text{max}}$ el valor propi més gran en mòdul i $\lambda_{\text{min}}$ el valor propi de més petit en mòdul de $A$, ja que

$$
\|A\|_2 = |\lambda_{\text{max}}|,\qquad \|A^{-1}\|_2 = \frac{1}{|\lambda_{\text{min}}|}.
$$

#### Exemple introductori

Calculem el nombre de condició de la matriu de l'exemple introductori:

$$
A =
\begin{pmatrix}
9 & 9 & 5 & 0\\
0 & 9 & 3 & 8\\
7 & 3 & 8 & 8\\
6 & 1 & 8 & 9
\end{pmatrix},
$$

utilitzant diferents normes.

In [ ]:
import numpy as np
from numpy.linalg import cond

# Matriu del sistema
A = np.array([[9,9,5,0],[0,9,3,8],[7,3,8,8],[6,1,8,9]],dtype = np.float64)

# Nombre de condició en diferents normes:

# Frobenius:
muF = cond(A,'fro')

# Norma 1:
mu1 = cond(A,1)

# Norma infinit:
muinf = cond(A,np.inf)

# Norma euclidiana:
mu2 = cond(A,2) # També es pot fer directament cond(A)


# Printem els resultats:
print("  Cond_F(A) =", muF, " (Norma Frobenius)")
print("  Cond_1(A) =",mu1, " (Norma 1)")
print("Cond_inf(A) =",muinf, " (Norma infinit)")
print("  Cond_2(A) =",mu2, " (Norma Euclidiana)")

  Cond_F(A) = 49569.2985929785  (Norma Frobenius)
  Cond_1(A) = 59824.9999999983  (Norma 1)
Cond_inf(A) = 59383.99999999832  (Norma infinit)
  Cond_2(A) = 43594.2272667525  (Norma Euclidiana)


&nbsp;

### Propagació de l'error a les dades

En general, quan volem resoldre el sistems lineal $Ax = b$ no disposem exctament dels valors de la matriu $A$ i del vector $b$, si no que tenim una certa aproximació $A+\delta A$ i $b+\delta b$, i per tant el que volem trobar és una solució del sistema pertorbat:

$$
    (A + \delta A)(x+\delta x) = b + \delta b.
$$

Per aquests casos, tenim la següent fòrmula que ens permet fitar, de forma relativa, en una norma qualsevol el vector de l'error $\delta x$:

$$
    \frac{\|\delta x \|}{\| x \|} \leq \mu(A) \left(\frac{\|\delta b\|}{\|b\|} + \frac{\|\delta A\|}{\|A\|}\frac{\| x + \delta x\|}{\| x\|}\right).
$$

Fixem-nos que el nombre de condició apareix multiplicant la fita, i per tant, quan aquest sigui gran la fita serà molt dolenta. Per aquest motiu, quan $\mu(A)$ és gran parlem de matrius mal condicionades.

#### Demostració:

Fixem-nos primer de tot que:

$$
 \color{blue}{b} \color{black} + \delta b = (A + \delta A)(x+\delta x) = \color{blue}{Ax}\color{black} + A\delta x + \delta A(x+\delta x)
$$

utilitzant ara que $Ax = b$, tenim:

$$
\begin{aligned}
    \delta b = A\delta x + \delta A(x+\delta x) & \Longleftrightarrow A\delta x = \delta b - \delta A(x+\delta x)\\
    & \Longleftrightarrow \delta x = A^{-1}\left[\delta b - \delta A(x+\delta x)\right].
\end{aligned}
$$



Si prenem normes a l'última igualtat, tenim:

$$
\begin{aligned}
    \|\delta x\| &= \|A^{-1}\left[\delta b - \delta A(x+\delta x)\right]\|\\
    &\leq \|A^{-1}\|\,\|\delta b - \delta A(x+\delta x)\|\\
    & \leq \|A^{-1}\|\,\left(\|\delta b\| + \|\delta A(x+\delta x)\|\right)\\
    & \leq \|A^{-1}\|\,\left(\|\delta b\| + \|\delta A\|\,\|x+\delta x\|\right)
\end{aligned}    
$$

Dividint ara a banda i banda per la norma de $x$, i utilitzant un altre cop que $Ax = b$ i que per tant
$$
\|A\|\,\|x\| \geq \|Ax\| = \|b\| \Longleftrightarrow \|x\| \geq \frac{\|b\|}{\|A\|}
$$
tenim:

$$
\begin{aligned}
\frac{\|\delta x\|}{\|x\|} & \leq \frac{\|A^{-1}\|\,\left(\|\delta b\| + \|\delta A\|\,\|x+\delta x\|\right)}{\|x\|}\\
& = \|A^{-1}\|\,\left(\frac{\|\delta b\|}{\|x\|} + \|\delta A\|\,\frac{\|x+\delta x\|}{\|x\|}\right)\\
& = \|A^{-1}\|\,\left(\frac{\|\delta b\|}{\color{orange}{\|x\|}} + \color{blue}{\|A\|}\frac{\|\delta A\|}{\color{blue}{\|A\|}}\,\frac{\|x+\delta x\|}{\|x\|}\right)\\
& \leq \|A^{-1}\|\,\left(\color{orange}{\|A\|}\frac{\|\delta b\|}{\color{orange}{\|b\|}} + \|A\|\frac{\|\delta A\|}{\|A\|}\,\frac{\|x+\delta x\|}{\|x\|}\right)\\
& = \|A^{-1}\|\,\|A\|\left(\frac{\|\delta b\|}{\|b\|} + \frac{\|\delta A\|}{\|A\|}\,\frac{\|x+\delta x\|}{\|x\|}\right)\\
& = \mu(A) \left(\frac{\|\delta b\|}{\|b\|} + \frac{\|\delta A\|}{\|A\|}\,\frac{\|x+\delta x\|}{\|x\|}\right)\\
\end{aligned}
$$


&nbsp;

I per tant hem demostrat la fita $\square$.

&nbsp;

### *Exercici 18*

*Partint de*

$$
(A+\delta A)(x+\delta x)=b+\delta b,
$$

*demostreu que si $||A^{-1}||\,||\delta A|| <1$, llavors*

$$ \frac{\|\delta x\|}{\|x\|}\le\frac{\mu(A)}{1-\mu(A)\frac{\|\delta A\|}{\|A\|}}\left(\frac{\|\delta b\|}{\|b\|}+\frac{\|\delta A\|}{\|A\|}\right),
$$

*on $\mu(A)$ és el nombre de condició de matriu $A$.*

#### Resolució:

Utilitzarem la fita que hem demostrat anteriorment, tenim:

$$
\begin{aligned}
\frac{\|\delta x \|}{\| x \|} &\leq \mu(A) \left(\frac{\|\delta b\|}{\|b\|} + \frac{\|\delta A\|}{\|A\|}\frac{\color{blue}{\| x + \delta x\|}}{\| x\|}\right)\\
&\leq \mu(A) \left(\frac{\|\delta b\|}{\|b\|} + \frac{\|\delta A\|}{\|A\|}\frac{\color{blue}{\| x\| + \|\delta x\|}}{\| x\|}\right)\\
& = \mu(A) \left(\frac{\|\delta b\|}{\|b\|} + \frac{\|\delta A\|}{\|A\|} + \frac{\|\delta A\|}{\|A\|}\frac{\|\delta x\|}{\| x\|}\right)
\end{aligned}
$$

Així, aïllant la part en $||\delta x||/||x||$ tenim:

$$
\begin{aligned}
 \mu(A) \left(\frac{\|\delta b\|}{\|b\|} + \frac{\|\delta A\|}{\|A\|} \right) & \geq \frac{\|\delta x\|}{\| x\|} - \mu(A)\frac{\|\delta A\|}{\|A\|}\frac{\|\delta x\|}{\| x\|}\\
 & = \left(1 - \mu(A)\frac{\|\delta A\|}{\|A\|}\right)\frac{\|\delta x\|}{\| x\|}
\end{aligned}
$$

Utilitzant per últim que
$$
    \mu(A)\frac{\|\delta A\|}{\|A\|} = \|A^{-1}\|\|\delta A\| < 1,
$$
tenim:

$$ \frac{\|\delta x\|}{\|x\|}\le\frac{\mu(A)}{1-\mu(A)\frac{\|\delta A\|}{\|A\|}}\left(\frac{\|\delta b\|}{\|b\|}+\frac{\|\delta A\|}{\|A\|}\right),
$$

